# Transformer with independent quadratic post-processing

This notebook evaluates one normally trained visit-level Transformer followed by patient-wise quadratic least-squares fitting after prediction.

There is no second Transformer, no model blending, and **no backpropagation through the quadratic fitting**. The quadratic model uses only a patient's Transformer predictions and visit ages—never validation targets.

Evaluation preserves the patient-separated five-fold split and uses seeds 42, 43 and 44. Predictions are averaged across seeds before reporting the primary OOF score. The final-training cell is disabled by default (`RUN_FINAL = False`).

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.base import clone
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset

PROJECT_DIR = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "exploration/features_selection.ipynb").is_file()
)

# Reuse the authoritative preprocessing by stable cell ID; no experiment cells run here.
reference = json.loads((PROJECT_DIR / "exploration/features_selection.ipynb").read_text())
previous_dir = Path.cwd()
try:
    os.chdir(PROJECT_DIR)
    for cell_id in [
        "selection-imports", "selection-history", "selection-imputation",
        "selection-features", "selection-pipeline",
    ]:
        source = "".join(next(c for c in reference["cells"] if c.get("id") == cell_id)["source"])
        exec(source)
finally:
    os.chdir(previous_dir)

DATA_DIR = PROJECT_DIR / "data"
PREPROCESSING = clone(pipeline[:2])
SEEDS = (42, 43, 44)
SPLIT_SEED = 42
CALIBRATION_FOLD = 1
VALIDATION_FOLDS = (2, 3, 4, 5)
EPOCHS, BATCH_SIZE = 160, 128
LEARNING_RATE = 0.004528032053597474
WEIGHT_DECAY = 1.2983658687387707e-5
CACHE_DIR = DATA_DIR / "transformer_quadratic_oof"
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else "cpu"
)
torch.set_num_threads(3)
print(f"Device: {DEVICE}; calibration fold: {CALIBRATION_FOLD}; seeds: {SEEDS}")

Device: mps; calibration fold: 1; seeds: (42, 43, 44)


## Method

For each training seed and outer fold:

1. train the original Transformer with ordinary MSE;
2. predict every visit of each held-out patient;
3. independently fit

\[
q(u)=a+bu+cu^2
\]

to that patient's predicted visits, with visit time `u` scaled to `[0, 1]`;
4. evaluate the quadratic predictions against the untouched targets.

The polynomial fit is NumPy post-processing and is completely outside the PyTorch computation graph.

In [2]:
def encoder_features(imputed):
    features = imputed.drop(columns="patient_id").copy()
    features["time_since_first_visit"] = (
        imputed["age"] - imputed.groupby("patient_id")["age"].transform("min")
    )
    return features


def sequence_dataset(raw, features, targets=None):
    visits = raw.reset_index(drop=True).sort_values(["patient_id", "age", "Index"])
    row_groups = [g.index.to_numpy() for _, g in visits.groupby("patient_id", sort=False)]
    values = torch.tensor(np.asarray(features), dtype=torch.float32)
    labels = torch.tensor(
        np.zeros(len(raw)) if targets is None else np.asarray(targets),
        dtype=torch.float32,
    )
    return TensorDataset(
        pad_sequence([values[rows] for rows in row_groups], batch_first=True),
        pad_sequence([labels[rows] for rows in row_groups], batch_first=True),
        pad_sequence([torch.tensor(rows) for rows in row_groups], batch_first=True, padding_value=-1),
    )


class VisitTransformer(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        width, dropout = 32, 0.20773426359864192
        self.projection = nn.Linear(n_features, width)
        layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=8, dim_feedforward=3 * width,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=5, enable_nested_tensor=False)
        for parameter in self.encoder.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        self.head = nn.Sequential(
            nn.Linear(width, 16), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(16, 16), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(16, 1),
        )

    def forward(self, values, padding):
        hidden = self.encoder(
            self.projection(values), src_key_padding_mask=padding, is_causal=False
        )
        return self.head(hidden).squeeze(-1)


def fit_transformer(raw, targets, preprocessing, seed):
    torch.manual_seed(seed)
    features = encoder_features(preprocessing.transform(raw))
    scaler = StandardScaler().fit(features)
    target_mean = targets.mean()
    target_scale = max(targets.std(ddof=0), 1e-8)
    training = sequence_dataset(
        raw, scaler.transform(features), (targets - target_mean) / target_scale
    )
    network = VisitTransformer(features.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(
        network.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=1, eta_min=LEARNING_RATE * 0.01
    )
    loader = DataLoader(
        training, batch_size=BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    for epoch in range(1, EPOCHS + 1):
        network.train()
        for values, labels, rows in loader:
            values, labels, rows = (x.to(DEVICE) for x in (values, labels, rows))
            valid = rows >= 0
            optimizer.zero_grad(set_to_none=True)
            prediction = network(values, ~valid)
            loss = nn.functional.mse_loss(prediction[valid], labels[valid])
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        if epoch % 40 == 0:
            print(f"seed={seed}, epoch={epoch}/{EPOCHS}", flush=True)
    network.requires_grad_(False).eval()
    return dict(
        network=network, preprocessing=preprocessing, scaler=scaler,
        target_mean=target_mean, target_scale=target_scale,
    )


@torch.no_grad()
def predict_transformer(fitted, raw):
    features = encoder_features(fitted["preprocessing"].transform(raw))
    data = sequence_dataset(raw, fitted["scaler"].transform(features))
    prediction = np.empty(len(raw), dtype=np.float32)
    for values, _, rows in DataLoader(data, batch_size=BATCH_SIZE):
        values, rows = values.to(DEVICE), rows.to(DEVICE)
        valid = rows >= 0
        output = fitted["network"](values, ~valid)
        prediction[rows[valid].cpu().numpy()] = output[valid].cpu().numpy()
    return prediction * fitted["target_scale"] + fitted["target_mean"]


def project_quadratic(raw, prediction):
    # Pure post-processing: no target values and no PyTorch computation graph.
    frame = raw[["patient_id", "age", "Index"]].copy()
    frame["prediction"] = np.asarray(prediction)
    projected = pd.Series(index=raw.index, dtype=float)
    for _, visits in frame.sort_values(["patient_id", "age", "Index"]).groupby("patient_id"):
        age = visits["age"].to_numpy()
        u = (age - age[0]) / max(age[-1] - age[0], 1e-8)
        coefficients = np.polyfit(u, visits["prediction"].to_numpy(), 2)
        projected.loc[visits.index] = np.polyval(coefficients, u)
    return projected.loc[raw.index].to_numpy()

## Out-of-fold predictions

One Transformer is fitted for every `(seed, fold)` pair: 15 fits in total. Raw and quadratically post-processed predictions are cached together, allowing interrupted runs to resume.

In [3]:
RUN_CV = True

splits = list(GroupKFold(
    n_splits=5, shuffle=True, random_state=SPLIT_SEED
).split(X, y, groups=X["patient_id"]))


def fit_fold(seed, fold):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    path = CACHE_DIR / f"seed_{seed}_fold_{fold}.csv"
    if path.is_file():
        print(f"Loading {path.name}")
        return pd.read_csv(path)

    train_index, valid_index = splits[fold - 1]
    raw_train, raw_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]
    preprocessing = clone(PREPROCESSING).fit(raw_train, y_train)
    with threadpool_limits(limits=3):
        model = fit_transformer(raw_train, y_train, preprocessing, seed)
        raw_prediction = predict_transformer(model, raw_valid)
    quadratic_prediction = project_quadratic(raw_valid, raw_prediction)

    result = raw_valid[["Index", "patient_id", "age"]].assign(
        row_position=valid_index, fold=fold, seed=seed, target=y_valid.to_numpy(),
        raw=raw_prediction, quadratic=quadratic_prediction,
    )
    result.to_csv(path, index=False)
    print(
        f"Saved {path.name}: raw={mean_squared_error(y_valid, raw_prediction):.6f}, "
        f"quadratic={mean_squared_error(y_valid, quadratic_prediction):.6f}"
    )
    return result


if RUN_CV:
    oof = pd.concat(
        [fit_fold(seed, fold) for seed in SEEDS for fold in range(1, 6)],
        ignore_index=True,
    )
    assert oof.groupby(["seed", "patient_id"])["fold"].nunique().eq(1).all()
    assert len(oof) == len(X) * len(SEEDS)
else:
    print("Set RUN_CV=True to train or load all OOF predictions.")

/var/folders/q8/d21qf_z93f37d2z9b04pqw3h0000gn/T/ipykernel_53768/2099943157.py:18: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:214.)
  pad_sequence([values[rows] for rows in row_groups], batch_first=True),


seed=42, epoch=40/160
seed=42, epoch=80/160
seed=42, epoch=120/160
seed=42, epoch=160/160
Saved seed_42_fold_1.csv: raw=6.778061, quadratic=6.635020
seed=42, epoch=40/160
seed=42, epoch=80/160
seed=42, epoch=120/160
seed=42, epoch=160/160
Saved seed_42_fold_2.csv: raw=6.860712, quadratic=6.737121
seed=42, epoch=40/160
seed=42, epoch=80/160
seed=42, epoch=120/160
seed=42, epoch=160/160
Saved seed_42_fold_3.csv: raw=6.656414, quadratic=6.529014
seed=42, epoch=40/160
seed=42, epoch=80/160
seed=42, epoch=120/160
seed=42, epoch=160/160
Saved seed_42_fold_4.csv: raw=6.994601, quadratic=6.850154
seed=42, epoch=40/160
seed=42, epoch=80/160
seed=42, epoch=120/160
seed=42, epoch=160/160
Saved seed_42_fold_5.csv: raw=6.758803, quadratic=6.587485
seed=43, epoch=40/160
seed=43, epoch=80/160
seed=43, epoch=120/160
seed=43, epoch=160/160
Saved seed_43_fold_1.csv: raw=6.894592, quadratic=6.727866
seed=43, epoch=40/160
seed=43, epoch=80/160
seed=43, epoch=120/160
seed=43, epoch=160/160
Saved seed_43_fo

## Performance and paired uncertainty

The primary comparison uses predictions averaged over seeds 42, 43 and 44. MSE is visit-weighted, while confidence intervals and the sign-flip test preserve patient clusters.

In [4]:
def cluster_inference(frame, n_bootstrap=10_000, n_permutations=20_000, seed=20261108):
    scored = frame.assign(
        raw_error=(frame["target"] - frame["raw"]) ** 2,
        quadratic_error=(frame["target"] - frame["quadratic"]) ** 2,
    )
    clusters = scored.groupby("patient_id").agg(
        raw_sum=("raw_error", "sum"), quadratic_sum=("quadratic_error", "sum"),
        n=("target", "size"),
    )
    rng = np.random.default_rng(seed)
    bootstrap_gain, bootstrap_relative = [], []
    for _ in range(n_bootstrap):
        sample = clusters.iloc[rng.integers(0, len(clusters), len(clusters))]
        raw_mse = sample["raw_sum"].sum() / sample["n"].sum()
        quadratic_mse = sample["quadratic_sum"].sum() / sample["n"].sum()
        bootstrap_gain.append(raw_mse - quadratic_mse)
        bootstrap_relative.append(100 * (1 - quadratic_mse / raw_mse))

    patient_delta = (clusters["raw_sum"] - clusters["quadratic_sum"]).to_numpy()
    observed = abs(patient_delta.sum())
    exceedances = sum(
        abs((rng.choice([-1, 1], len(patient_delta)) * patient_delta).sum()) >= observed
        for _ in range(n_permutations)
    )
    patient_mse = scored.groupby("patient_id")[["raw_error", "quadratic_error"]].mean()
    return pd.Series({
        "absolute_gain": scored["raw_error"].mean() - scored["quadratic_error"].mean(),
        "absolute_gain_CI_low": np.quantile(bootstrap_gain, 0.025),
        "absolute_gain_CI_high": np.quantile(bootstrap_gain, 0.975),
        "relative_gain_percent": 100 * (1 - scored["quadratic_error"].mean() / scored["raw_error"].mean()),
        "relative_gain_CI_low": np.quantile(bootstrap_relative, 0.025),
        "relative_gain_CI_high": np.quantile(bootstrap_relative, 0.975),
        "patient_win_rate": (patient_mse["quadratic_error"] < patient_mse["raw_error"]).mean(),
        "cluster_sign_flip_pvalue": (exceedances + 1) / (n_permutations + 1),
    })


if RUN_CV:
    seed_average = (
        oof.groupby(["row_position", "Index", "patient_id", "age", "fold", "target"], as_index=False)
        [["raw", "quadratic"]].mean()
    )
    fold_scores = seed_average.groupby("fold").apply(
        lambda part: pd.Series({
            "raw_MSE": mean_squared_error(part["target"], part["raw"]),
            "quadratic_MSE": mean_squared_error(part["target"], part["quadratic"]),
        }),
        include_groups=False,
    )
    fold_scores["improvement_percent"] = 100 * (
        1 - fold_scores["quadratic_MSE"] / fold_scores["raw_MSE"]
    )

    seed_scores = oof.groupby("seed").apply(
        lambda part: pd.Series({
            "raw_MSE": mean_squared_error(part["target"], part["raw"]),
            "quadratic_MSE": mean_squared_error(part["target"], part["quadratic"]),
        }),
        include_groups=False,
    )
    seed_scores["improvement_percent"] = 100 * (
        1 - seed_scores["quadratic_MSE"] / seed_scores["raw_MSE"]
    )

    summary = pd.Series({
        "raw_MSE": mean_squared_error(seed_average["target"], seed_average["raw"]),
        "quadratic_MSE": mean_squared_error(seed_average["target"], seed_average["quadratic"]),
    })
    summary["improvement_percent"] = 100 * (
        1 - summary["quadratic_MSE"] / summary["raw_MSE"]
    )
    inference = cluster_inference(seed_average)

    display(summary.to_frame("value"))
    display(fold_scores)
    print("Seed sensitivity:")
    display(seed_scores)
    print("Patient-cluster paired inference:")
    display(inference.to_frame("value"))

,value
raw_MSE,6.490546
quadratic_MSE,6.384694
improvement_percent,1.630863


,raw_MSE,quadratic_MSE,improvement_percent
fold,,,
1,6.468149,6.357047,1.717681
2,6.489532,6.389192,1.546190
3,6.336805,6.241314,1.506930
4,6.779506,6.668436,1.638309
5,6.372113,6.261021,1.743398


Seed sensitivity:


,raw_MSE,quadratic_MSE,improvement_percent
seed,,,
42,6.810680,6.668739,2.084090
43,6.864702,6.708763,2.271606
44,6.852406,6.709468,2.085954


Patient-cluster paired inference:


,value
absolute_gain,0.105852
absolute_gain_CI_low,0.098010
absolute_gain_CI_high,0.114044
relative_gain_percent,1.630863
relative_gain_CI_low,1.484263
relative_gain_CI_high,1.786414
patient_win_rate,0.772629
cluster_sign_flip_pvalue,0.000050


## Final fit and test prediction

This cell fits one Transformer on all training patients for each seed, averages their raw test predictions, applies one patient-wise quadratic projection, and writes the submission. It is intentionally disabled.

In [5]:
RUN_FINAL = False

if RUN_FINAL:
    X_test = pd.read_csv(DATA_DIR / "X_test_oiZ2ukx.csv")[X.columns]
    full_preprocessing = clone(PREPROCESSING).fit(X, y)
    raw_predictions = []
    with threadpool_limits(limits=3):
        for seed in SEEDS:
            model = fit_transformer(X, y, full_preprocessing, seed)
            raw_predictions.append(predict_transformer(model, X_test))

    raw_prediction = np.mean(raw_predictions, axis=0)
    final_prediction = project_quadratic(X_test, raw_prediction)
    assert np.isfinite(final_prediction).all()

    submission = pd.DataFrame({"Index": X_test["Index"], "target": final_prediction})
    submission_path = DATA_DIR / "submission_transformer_quadratic.csv"
    submission.to_csv(submission_path, index=False)
    print(f"Saved {submission_path} ({len(submission):,} visits)")